# Reinterpreting Blair et al. (2013): What the Data Actually Show

Blair et al. (2013) conducted endorsement experiments in Pakistan to measure attitudes toward militant groups. Their interpretation of the results focuses on the negative sign of treatment effects, concluding that Pakistanis hold these groups in "low regard."

**This notebook shows that interpretation is wrong.**

With high baseline policy support (~80%), small negative treatment effects are actually consistent with *majority support* for the endorsing groups. The data suggest ~79% of Pakistanis may support militant organizations, not the opposite.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("husl")

from endorsement_analysis_notebook import EndorsementAnalyzer
from endorsement_viz import plot_bounds_comparison, plot_sensitivity_analysis

## 1. Blair et al.'s Findings and Their Interpretation

### The Data
Blair et al. report:
- **Baseline policy support** (constant term): ~0.80 (80% support policies in control condition)
- **Treatment effects**: Small negative, ranging from -0.008 to -0.015 depending on the group
- "Strongest" effect: Afghan Taliban at -0.015 (1.5 percentage points)

### Their Interpretation

> "We find that Pakistanis in general are **weakly negative** toward Islamist militant organizations, as shown in Table 1."

> "The coefficients are negative and statistically significant at the 10% level for all but the sectarian tanzeems, suggesting that **Pakistanis hold militant groups in low regard**."

> "The effect is statistically and substantively strongest for the Afghan Taliban, where the treatment reduces support by 1.5%, roughly 10% of a standard deviation in support for policies in the control group."

### The Problem
This interpretation confuses the **direction** of the effect with the **level** of support. A small negative coefficient with high baseline does NOT imply opposition.

In [ ]:
# Blair et al. parameters
ATE = -0.011
BASELINE = 0.80
ATE_STRONGEST = -0.015  # Afghan Taliban

print("Blair et al. (2013) Key Parameters")
print("=" * 40)
print(f"Baseline policy support (control): {BASELINE:.1%}")
print(f"Average treatment effect: {ATE:.3f}")
print(f"Strongest effect (Afghan Taliban): {ATE_STRONGEST:.3f}")
print(f"Treatment group average: {BASELINE + ATE:.1%}")

## 2. What Values Are Consistent With The Data?

We now show the range of supporter proportions that are consistent with Blair et al.'s observed data, under different assumptions.

In [ ]:
analyzer = EndorsementAnalyzer(ate=ATE, constant_term=BASELINE)

# Method 1: Binary framework (no baseline info)
binary = analyzer.binary_framework_bounds()

# Method 2: Baseline analysis (exact solution)
baseline_analysis = analyzer.baseline_support_analysis()

# Method 3: Continuous effects bounds
continuous = analyzer.continuous_bounds()

print("Support Estimates by Method")
print("=" * 50)
print(f"\n1. Binary Framework (ignoring baseline):")
print(f"   Maximum supporters: {binary['p_max']:.1%}")
print(f"\n2. Baseline Analysis (using constant term):")
print(f"   Proportion of supporters: {baseline_analysis['proportion_supporters']:.1%}")
print(f"\n3. Continuous Effects:")
print(f"   Extreme scenario (opponents react maximally): {continuous['p_max_extreme']:.1%}")
print(f"   Symmetric small effects: {continuous['p_symmetric']:.1%}")

In [ ]:
# Visualization: Comparing methods
fig, ax = plt.subplots(figsize=(10, 6))

methods = ['Binary\n(No Baseline)', 'Baseline Analysis\n(Exact)', 'Continuous\n(Extreme)', 'Continuous\n(Symmetric)']
estimates = [
    binary['p_max'],
    baseline_analysis['proportion_supporters'],
    continuous['p_max_extreme'],
    continuous['p_symmetric']
]
colors = ['#a8d5e5', '#f4a460', '#90ee90', '#fffacd']

bars = ax.bar(methods, estimates, color=colors, edgecolor='black', linewidth=1.5)

ax.axhline(0.5, color='red', linestyle='--', linewidth=2, label='50% (Majority threshold)')
ax.set_ylabel('Proportion Supporting Militant Groups', fontsize=12)
ax.set_title('Support Estimates Under Different Assumptions\nBlair et al. (2013): ATE = -0.011, Baseline = 0.80', fontsize=14)
ax.set_ylim(0, 1)

for bar, est in zip(bars, estimates):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
            f'{est:.1%}', ha='center', va='bottom', fontsize=12, fontweight='bold')

ax.legend(loc='lower right')
plt.tight_layout()
plt.show()

### Table: All Groups from Blair et al. Table 1

Let's reinterpret all groups using the baseline analysis approach.

In [ ]:
# Approximate ATEs from Blair et al. Table 1 (Panel A)
groups = {
    'Afghan Taliban': -0.015,
    'Pakistani Taliban (TTP)': -0.011,
    'Kashmiri Militants': -0.010,
    'Sectarian Groups': -0.005,
    'Average': -0.011
}

results = []
for group, ate in groups.items():
    p_binary = (ate + 1) / 2
    p_baseline = BASELINE + ate
    results.append({
        'Group': group,
        'ATE': ate,
        'Blair Interpretation': 'Negative/Low Regard',
        'Binary (no baseline)': f"{p_binary:.1%}",
        'With Baseline (actual)': f"{p_baseline:.1%}",
        'Corrected Interpretation': 'Majority Support'
    })

df = pd.DataFrame(results)
print("Reinterpretation of Blair et al. Table 1")
print("=" * 90)
print(df.to_string(index=False))

## 3. The Interpretation Problem: Why Blair et al. Got It Wrong

### The Mechanics

Under the binary switching model:
- **Supporters** (proportion $p$): Always support the policy when their favored group endorses it (outcome = 1 in treatment)
- **Opponents** (proportion $1-p$): Always oppose the policy when the disliked group endorses it (outcome = 0 in treatment)

Therefore:
$$\text{Treatment group average} = p \cdot 1 + (1-p) \cdot 0 = p$$

So $p = \alpha + \beta = 0.80 + (-0.011) = 0.789$

**78.9% of respondents support the militant groups.**

In [ ]:
# Visual explanation
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: The mechanics
ax1 = axes[0]
categories = ['Control Group\n(No Endorsement)', 'Treatment Group\n(With Endorsement)']
control_support = BASELINE
treatment_support = BASELINE + ATE

bars = ax1.bar(categories, [control_support, treatment_support], 
               color=['lightblue', 'salmon'], edgecolor='black', linewidth=1.5)
ax1.set_ylabel('Average Policy Support', fontsize=12)
ax1.set_title('Control vs Treatment Group Averages', fontsize=14)
ax1.set_ylim(0.7, 0.85)

for bar, val in zip(bars, [control_support, treatment_support]):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
            f'{val:.1%}', ha='center', va='bottom', fontsize=12, fontweight='bold')

ax1.annotate('', xy=(1, treatment_support), xytext=(0, control_support),
            arrowprops=dict(arrowstyle='<->', color='red', lw=2))
ax1.text(0.5, (control_support + treatment_support)/2, f'ATE = {ATE:.3f}',
        ha='center', va='center', fontsize=11,
        bbox=dict(boxstyle='round', facecolor='yellow', alpha=0.8))

# Right: The interpretation
ax2 = axes[1]
p_supporters = treatment_support
categories2 = ['Supporters\n(of militant groups)', 'Opponents']
proportions = [p_supporters, 1 - p_supporters]
colors2 = ['#ff6b6b', '#4ecdc4']

bars2 = ax2.bar(categories2, proportions, color=colors2, edgecolor='black', linewidth=1.5)
ax2.set_ylabel('Proportion of Population', fontsize=12)
ax2.set_title('Inferred Support for Militant Groups\n(Under Binary Switching Model)', fontsize=14)
ax2.set_ylim(0, 1)

for bar, val in zip(bars2, proportions):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
            f'{val:.1%}', ha='center', va='bottom', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()

### The Key Insight

Blair et al.'s error: They interpret the **sign** of the ATE as reflecting the **level** of support.

| What the negative ATE tells you | What Blair et al. conclude |
|--------------------------------|---------------------------|
| Slightly more opponents than supporters among the "swing" group | "Weakly negative toward militant groups" |
| The balance of switchers tilts slightly toward opposition | "Hold militant groups in low regard" |

**The correct interpretation:**
- The treatment group average (78.9%) equals the proportion of supporters
- A small negative ATE with high baseline means **most people support the group**
- The data are consistent with **78.9% of Pakistanis supporting militant groups**

In [ ]:
print("THE KEY INSIGHT")
print("=" * 60)
print("\nBlair et al.'s interpretation:")
print('  "Pakistanis hold militant groups in low regard"')
print("\nWhat the data actually show:")
print(f"  78.9% of Pakistanis support militant groups")
print("\nWhy the discrepancy?")
print("  - Negative sign reflects BALANCE of switchers, not LEVEL of support")
print("  - High baseline (80%) + small negative (-1.1%) = 78.9% supporters")
print("  - The negative sign means slightly more opponents than supporters")
print("    among those who switch, but the LEVEL is still a large majority")

## 4. Sensitivity Analysis

How do support estimates change as we vary the ATE and baseline?

In [ ]:
# Sensitivity to ATE
ate_range = np.linspace(-0.10, 0.10, 100)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: Varying ATE with fixed baseline
ax1 = axes[0]
p_binary = (ate_range + 1) / 2
p_baseline = BASELINE + ate_range

ax1.plot(ate_range, p_binary, 'b-', linewidth=2, label='Binary (no baseline)')
ax1.plot(ate_range, p_baseline, 'r-', linewidth=2, label='With baseline = 0.80')
ax1.axvline(ATE, color='green', linestyle='--', linewidth=2, alpha=0.7, label=f'Observed ATE = {ATE}')
ax1.axhline(0.5, color='gray', linestyle=':', linewidth=1, alpha=0.7)

ax1.fill_between(ate_range, 0.5, p_baseline, where=(p_baseline > 0.5), 
                 alpha=0.3, color='red', label='Majority support region')

ax1.set_xlabel('Average Treatment Effect', fontsize=12)
ax1.set_ylabel('Estimated Proportion Supporting Group', fontsize=12)
ax1.set_title('Sensitivity to ATE\n(Baseline fixed at 0.80)', fontsize=14)
ax1.legend(loc='lower right')
ax1.set_ylim(0, 1)
ax1.grid(True, alpha=0.3)

# Right: Varying baseline with fixed ATE
ax2 = axes[1]
baseline_range = np.linspace(0.5, 1.0, 100)
p_varying_baseline = baseline_range + ATE

ax2.plot(baseline_range, p_varying_baseline, 'r-', linewidth=2, label=f'ATE = {ATE}')
ax2.axvline(BASELINE, color='green', linestyle='--', linewidth=2, alpha=0.7, label=f'Observed baseline = {BASELINE}')
ax2.axhline(0.5, color='gray', linestyle=':', linewidth=1, alpha=0.7)

ax2.fill_between(baseline_range, 0.5, p_varying_baseline, where=(p_varying_baseline > 0.5),
                 alpha=0.3, color='red', label='Majority support region')

ax2.set_xlabel('Baseline Support (Constant Term)', fontsize=12)
ax2.set_ylabel('Estimated Proportion Supporting Group', fontsize=12)
ax2.set_title('Sensitivity to Baseline\n(ATE fixed at -0.011)', fontsize=14)
ax2.legend(loc='lower right')
ax2.set_ylim(0, 1)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Heatmap: Support estimates for different ATE and baseline combinations
ate_vals = np.linspace(-0.05, 0.05, 20)
baseline_vals = np.linspace(0.6, 0.95, 20)

support_matrix = np.zeros((len(baseline_vals), len(ate_vals)))
for i, b in enumerate(baseline_vals):
    for j, a in enumerate(ate_vals):
        support_matrix[i, j] = b + a

fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(support_matrix, cmap='RdYlGn', aspect='auto', 
               extent=[ate_vals[0], ate_vals[-1], baseline_vals[0], baseline_vals[-1]],
               origin='lower', vmin=0.4, vmax=1.0)

ax.axhline(BASELINE, color='white', linestyle='--', linewidth=2)
ax.axvline(ATE, color='white', linestyle='--', linewidth=2)
ax.plot(ATE, BASELINE, 'ko', markersize=15, markerfacecolor='white', markeredgewidth=2)
ax.annotate('Blair et al.', xy=(ATE, BASELINE), xytext=(ATE+0.02, BASELINE+0.05),
           fontsize=12, fontweight='bold',
           arrowprops=dict(arrowstyle='->', color='black'))

# Add contour for 50% line
contour = ax.contour(ate_vals, baseline_vals, support_matrix, levels=[0.5], colors='black', linewidths=2)
ax.clabel(contour, fmt='50%', fontsize=10)

ax.set_xlabel('Average Treatment Effect', fontsize=12)
ax.set_ylabel('Baseline Support', fontsize=12)
ax.set_title('Inferred Support Proportion\n(Green = High Support, Red = Low Support)', fontsize=14)

cbar = plt.colorbar(im, ax=ax)
cbar.set_label('Proportion Supporting Group', fontsize=12)

plt.tight_layout()
plt.show()

## 5. Alternative Scenarios & Assumptions

### Partial Awareness
What if only some fraction of respondents have genuine opinions about militant groups?

In [ ]:
# Partial awareness model
# ATE = q * (2p - 1) where q = fraction with opinions, p = proportion of opinion-holders who support

q_values = [0.5, 0.6, 0.7, 0.8, 0.9, 1.0]

print("Support Among Opinion-Holders (Partial Awareness Model)")
print("=" * 55)
print(f"{'Fraction with opinions (q)':<30} {'Support proportion (p)':<25}")
print("-" * 55)

for q in q_values:
    # p = ATE / (2q) + 0.5
    p = ATE / (2 * q) + 0.5
    print(f"{q:<30.0%} {p:<25.1%}")

print("\nNote: Even at 50% awareness, 48.9% of opinion-holders support the groups.")
print("      At 100% awareness, 49.45% support (binary framework without baseline).")

In [ ]:
# Heterogeneous effects simulation
np.random.seed(42)

n_simulations = 5
n_samples = 10000
std_values = [0.02, 0.05, 0.10, 0.15, 0.20]

print("Heterogeneous Effects Simulation (Normal Distribution)")
print("=" * 60)
print(f"{'Effect Std Dev':<20} {'Mean Positive Shifters':<25} {'95% CI':<20}")
print("-" * 60)

for std in std_values:
    positive_props = []
    for _ in range(n_simulations):
        effects = np.random.normal(ATE, std, n_samples)
        effects = np.clip(effects, -1, 1)
        effects = effects - np.mean(effects) + ATE  # Maintain ATE constraint
        positive_props.append(np.mean(effects > 0))
    
    mean_prop = np.mean(positive_props)
    ci_low = np.percentile(positive_props, 2.5)
    ci_high = np.percentile(positive_props, 97.5)
    print(f"{std:<20.2f} {mean_prop:<25.1%} [{ci_low:.1%}, {ci_high:.1%}]")

In [ ]:
# Visualize heterogeneous effects for one scenario
np.random.seed(42)
std = 0.10
effects = np.random.normal(ATE, std, 10000)
effects = np.clip(effects, -1, 1)
effects = effects - np.mean(effects) + ATE

fig, ax = plt.subplots(figsize=(10, 6))

ax.hist(effects, bins=50, alpha=0.7, color='lightblue', edgecolor='black', density=True)
ax.axvline(0, color='red', linestyle='--', linewidth=2, label='Zero effect')
ax.axvline(np.mean(effects), color='orange', linestyle='-', linewidth=2, 
           label=f'Mean = {np.mean(effects):.4f}')

positive_prop = np.mean(effects > 0)
ax.fill_between([0, max(effects)], 0, 0.1, alpha=0.3, color='green',
                transform=ax.get_xaxis_transform())

ax.text(0.15, 3, f'Positive effects:\n{positive_prop:.1%}', fontsize=12,
       bbox=dict(boxstyle='round', facecolor='lightgreen', alpha=0.8))

ax.set_xlabel('Individual Treatment Effect', fontsize=12)
ax.set_ylabel('Density', fontsize=12)
ax.set_title(f'Heterogeneous Effects Distribution\n(Mean = {ATE}, Std = {std})', fontsize=14)
ax.legend(loc='upper left')

plt.tight_layout()
plt.show()

## 6. External Validation

How do our estimates compare with direct polling?

In [ ]:
# External validation data
validation_data = {
    'Source': ['Pew Research (2015)', 'Pew Research (2015)', 'Blair et al. (2013)', 'Our Reinterpretation'],
    'Measure': ['Explicit TTP Support', 'Don\'t Know', 'Endorsement Exp (ATE)', 'Inferred Support'],
    'Value': ['9-14%', '30-49%', '-1.1%', '78.9%'],
    'Interpretation': [
        'Direct polling underestimates',
        'Many reluctant to express views',
        'Wrong: "low regard"',
        'Majority latent support'
    ]
}

df_validation = pd.DataFrame(validation_data)
print("External Validation")
print("=" * 90)
print(df_validation.to_string(index=False))

In [ ]:
# Visualization
fig, ax = plt.subplots(figsize=(10, 6))

sources = ['Pew: Explicit\nSupport', 'Pew: Don\'t\nKnow', 'Our\nReinterpretation']
low_vals = [0.09, 0.30, 0.789]
high_vals = [0.14, 0.49, 0.789]
midpoints = [(l+h)/2 for l, h in zip(low_vals, high_vals)]
errors = [(h-l)/2 for l, h in zip(low_vals, high_vals)]

colors = ['#4ecdc4', '#ffe66d', '#ff6b6b']
bars = ax.bar(sources, midpoints, yerr=errors, capsize=5, color=colors, 
              edgecolor='black', linewidth=1.5)

ax.axhline(0.5, color='gray', linestyle='--', linewidth=1.5, label='50% threshold')
ax.set_ylabel('Proportion', fontsize=12)
ax.set_title('Comparing Measures of Support for Militant Groups', fontsize=14)
ax.set_ylim(0, 1)

for i, (bar, val) in enumerate(zip(bars, midpoints)):
    if errors[i] > 0:
        label = f'{low_vals[i]:.0%}-{high_vals[i]:.0%}'
    else:
        label = f'{val:.1%}'
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + errors[i] + 0.03,
            label, ha='center', va='bottom', fontsize=11, fontweight='bold')

ax.legend()
plt.tight_layout()
plt.show()

### Reconciling the Gap

Why might endorsement experiments reveal higher support than direct polling?

1. **Social desirability bias**: Direct questions about militant support are sensitive
2. **High "Don't Know" rates**: 30-49% refuse to answer direct questions (Pew)
3. **Endorsement experiments**: Indirect measurement reduces bias
4. **Hidden preferences**: Many may support militants privately but not publicly

The large gap between 9-14% (explicit) and 78.9% (inferred) is consistent with substantial hidden support.

## 7. Conclusion

### Summary

Blair et al. (2013) interpret small negative treatment effects as evidence that Pakistanis hold militant groups in "low regard." This interpretation is incorrect.

**The data show:**
- Treatment group average: 78.9%
- Under binary switching: 78.9% support militant groups
- Even the "strongest" negative effect (Afghan Taliban, -1.5%) implies 78.5% support

**The interpretation error:**
- Negative coefficient sign ≠ majority opposition
- With high baseline, negative ATE means majority support
- The correct conclusion: **Most Pakistanis support militant groups**

**Key insight:**
The treatment group average, not the sign of the ATE, reveals the proportion of supporters.

In [ ]:
# Final summary visualization
fig, ax = plt.subplots(figsize=(12, 6))

interpretations = ['Blair et al.\nInterpretation', 'Our\nReinterpretation']
values = [0.25, 0.789]  # Approximate "low regard" vs actual 78.9%
colors = ['#e74c3c', '#27ae60']

bars = ax.bar(interpretations, values, color=colors, edgecolor='black', linewidth=2, width=0.5)

ax.axhline(0.5, color='gray', linestyle='--', linewidth=2, label='Majority threshold')
ax.set_ylabel('Implied Support for Militant Groups', fontsize=14)
ax.set_title('Blair et al. (2013): Two Interpretations of the Same Data\n'
             '(ATE = -0.011, Baseline = 0.80)', fontsize=16)
ax.set_ylim(0, 1)

ax.text(0, 0.30, '"Weakly negative"\n"Low regard"', ha='center', va='bottom', 
       fontsize=11, style='italic')
ax.text(1, 0.83, '78.9% support\n(Treatment avg)', ha='center', va='bottom',
       fontsize=11, fontweight='bold')

ax.annotate('', xy=(1, 0.789), xytext=(0, 0.25),
           arrowprops=dict(arrowstyle='->', color='black', lw=2,
                          connectionstyle='arc3,rad=0.3'))
ax.text(0.5, 0.55, 'Same data,\ndifferent interpretation', ha='center', va='center',
       fontsize=12, bbox=dict(boxstyle='round', facecolor='yellow', alpha=0.8))

ax.legend(loc='upper right')
plt.tight_layout()
plt.show()

print("\n" + "=" * 60)
print("BOTTOM LINE")
print("=" * 60)
print("\nBlair et al. conclude: 'Pakistanis hold militant groups in low regard'")
print("The data actually show: 78.9% of Pakistanis support militant groups")
print("\nThe negative sign of the ATE is misleading without baseline context.")